Build Deep Sequential Models & Training it from Scratch
=======================================================

# Setup

In [ ]:
import torch


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

In [ ]:
from copy import deepcopy
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import kagglehub
import datasets
import torch.nn as nn
import tokenizers
import transformers
import pandas as pd
import torchmetrics
from torchmetrics.classification import MultilabelAveragePrecision

torch.manual_seed(42)

In [ ]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

## Setup W&B for experiment tracking

In [ ]:
import wandb

wandb.login(key="your_wandb_API_Key")

# Prepare the data

## Get the data

In [ ]:
data_path = kagglehub.dataset_download("julian3833/jigsaw-toxic-comment-classification-challenge",
                                        "train.csv",
                                        output_dir="../data"
                                        )
dataset = datasets.load_dataset("csv", data_files=data_path)
dataset

## Text Normalization

In [ ]:
import re

# existing patterns
username_pattern = re.compile(r'\[\[user.*?\]\]|\[\[user talk.*?\]\]|user:\w+', re.IGNORECASE)
url_pattern = re.compile(r'http\S+|www\.\S+')  # Added URL pattern
whitespace_pattern = re.compile(r'\s+')
repeat_char_pattern = re.compile(r'(.)\1{2,}')

# wiki markup patterns
wiki_link_pattern = re.compile(r'\[\[(?:[^\]|]*\|)?([^\]]+)\]\]')
wiki_template_pattern = re.compile(r'\{\{.*?\}\}')
wiki_bold_italic_pattern = re.compile(r"'{2,5}")
wiki_heading_pattern = re.compile(r'=+\s*(.*?)\s*=+')

def preprocess_batch(batch):
    cleaned_texts = []

    for text in batch["comment_text"]:
        # Handle None or empty input
        if not text or not text.strip():
            cleaned_texts.append("[UNK]")  # Placeholder for empty texts
            continue

        text = text.lower()

        # remove usernames
        text = username_pattern.sub(" ", text)

        # Replace URLs with [UNK]
        text = url_pattern.sub("[UNK]", text)

        # convert [[link|text]] -> text
        text = wiki_link_pattern.sub(r"\1", text)

        # remove templates {{...}}
        text = wiki_template_pattern.sub(" ", text)

        # remove bold/italic markup
        text = wiki_bold_italic_pattern.sub("", text)

        # remove headings
        text = wiki_heading_pattern.sub(r"\1", text)

        # normalize stretched characters
        text = repeat_char_pattern.sub(r"\1\1", text)

        # normalize whitespace
        text = whitespace_pattern.sub(" ", text).strip()

        # ✅ Ensure text is not empty after cleaning
        if not text:
            text = "[UNK]"  # Fallback for texts that become empty

        cleaned_texts.append(text)

    batch["comment_text"] = cleaned_texts
    return batch

In [ ]:
dataset = dataset["train"].map(preprocess_batch, batched=True, num_proc=4)

## Tokenization

### Train a BBPE from scratch

#### Prepare the comments for training

In [ ]:
train = dataset.train_test_split(test_size=0.2, seed=42)["train"]
comments = [comment['comment_text'] for comment in train]
comments[:2]

#### Train the tokenizer

In [ ]:
special_tokens = ['<pad>', '<unk>']


bbpe_model = tokenizers.models.BPE(unk_token="<unk>")
bbpe_tokenizer = tokenizers.Tokenizer(bbpe_model)
bbpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
bbpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=30_000,
                                              special_tokens=special_tokens)
bbpe_tokenizer.train_from_iterator(comments, bbpe_trainer)

bbpe_tokenizer.enable_padding(
    pad_id=0,
    pad_token="<pad>"
)

#### Wrap our tokenizer in a HuggingFace PreTrainedTokenizerFast for easy integration

In [ ]:
tokenizer = transformers.PreTrainedTokenizerFast(
    tokenizer_object=bbpe_tokenizer
)

#### Testing our tokenizer

In [ ]:
encodings = tokenizer(comments[:2], truncation=True, max_length=256)

encodings

In [ ]:
encodings["input_ids"]

#### Use our tokenizer

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=256,
        padding=False
    )

In [ ]:
dataset = dataset.map(
    tokenize_batch,
    batched=True,
    num_proc=4
)
dataset = dataset.remove_columns(["comment_text", "id"])

dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate"
    ]
)

## Split data

In [ ]:
from transformers import DataCollatorWithPadding


LABEL_COLUMNS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Create a data collator that handles padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def collate_fn(batch):
    # Extract text features for padding
    text_features = [{
        "input_ids": item["input_ids"],
        "attention_mask": item["attention_mask"]
    } for item in batch]

    # Pad the batch
    padded = data_collator(text_features)

    # Extract labels
    labels = torch.stack([
        torch.tensor([item[label] for label in LABEL_COLUMNS], dtype=torch.float32)
        for item in batch
    ])

    X = {
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"]
    }

    return X, labels

In [ ]:
split = dataset.train_test_split(test_size=0.2, seed=42)
train_set = split["train"]
test_valid_set = split["test"]
split = test_valid_set.train_test_split(test_size=0.5, seed=42)
valid_set = split["train"]
test_set = split["test"]

### Splits statistics

In [ ]:
print(f"Train Shape: {train_set.shape}")
print(f"Valid Shape: {valid_set.shape}")
print(f"Test Shape: {test_set.shape}")
print("------------------------------------------------")

def calculate_label_statistics(dataset, split_name):
    """Calculate label statistics for a given dataset split."""

    # Convert to pandas DataFrame for easier manipulation
    df = dataset.to_pandas()

    # Extract label columns
    label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    labels = df[label_cols]

    # Calculate statistics
    stats = {}

    # 1. Number of rows without any toxic label (all labels are 0)
    stats['no_toxic_labels'] = (labels.sum(axis=1) == 0).sum()

    # 2. Number of rows with at least one toxic label
    stats['at_least_one_toxic'] = (labels.sum(axis=1) > 0).sum()

    # 3. For each label: number of rows where it takes value 1
    for col in label_cols:
        stats[f'{col}_count'] = labels[col].sum()

    # Add total rows for reference
    stats['total_rows'] = len(df)

    return stats

# Calculate statistics for each split
train_stats = calculate_label_statistics(train_set, "Train")
valid_stats = calculate_label_statistics(valid_set, "Valid")
test_stats = calculate_label_statistics(test_set, "Test")

# Create a summary DataFrame
summary_stats = pd.DataFrame({
    'Train': train_stats,
    'Valid': valid_stats,
    'Test': test_stats
}).T

# Display the results
print("=== Label Statistics Summary ===")
print(summary_stats)

# Calculate percentages for better understanding
print("\n=== Label Distribution Percentages ===")
percentage_stats = summary_stats.copy()
for split in ['Train', 'Valid', 'Test']:
    total = summary_stats.loc[split, 'total_rows']
    percentage_stats.loc[split, 'no_toxic_labels_pct'] = (summary_stats.loc[split, 'no_toxic_labels'] / total) * 100
    percentage_stats.loc[split, 'at_least_one_toxic_pct'] = (summary_stats.loc[split, 'at_least_one_toxic'] / total) * 100

    for col in ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']:
        percentage_stats.loc[split, f'{col}_pct'] = (summary_stats.loc[split, f'{col}_count'] / total) * 100

# Display percentage columns
percentage_cols = ['no_toxic_labels_pct', 'at_least_one_toxic_pct'] + [f'{col}_pct' for col in ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]
print(percentage_stats[percentage_cols].round(2))

VOCAB_SIZE = len(tokenizer)
print(f"Vocabulary size: {VOCAB_SIZE}")


### Positive class weights

In [ ]:
# compute per-label positive/negative counts on the training split
df_train = train_set.to_pandas()
label_counts = df_train[LABEL_COLUMNS].sum()
total_examples = len(df_train)

neg_counts = total_examples - label_counts

# pos_weight is commonly used with BCEWithLogitsLoss for multi-label tasks
pos_weight = (neg_counts / (label_counts + 1e-9)).astype("float32")

print("Label counts (pos):")
print(label_counts)
print("\nPos weights (neg/pos):")
print(pos_weight)

# store as torch tensor for passing to loss
pos_weight_tensor = torch.tensor(pos_weight.values, dtype=torch.float32).to(device)
pos_weight_tensor

## Load the datasets into PyTorch DataLoaders

In [ ]:
BATCH_SIZE = 256

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
# Add after creating dataloaders
sample_batch = next(iter(train_loader))
X_sample, y_sample = sample_batch

print("Input shapes:")
print(f"  input_ids: {X_sample['input_ids'].shape}")
print(f"  attention_mask: {X_sample['attention_mask'].shape}")
print(f"  labels: {y_sample.shape}")
print(f"  labels dtype: {y_sample.dtype}")

assert X_sample['input_ids'].shape[0] == BATCH_SIZE
assert y_sample.shape == (BATCH_SIZE, 6)
assert y_sample.dtype == torch.float32

# Train & Eval functions

In [ ]:
!pip install -q torchmetrics

In [ ]:
import torchmetrics
from torchmetrics.classification import MultilabelAveragePrecision

def evaluate_tm(model, data_loader, metric):

    model.eval()
    metric.reset()

    with torch.no_grad():
        for X_batch, y_batch in data_loader:

            X_batch = {k: v.to(device) for k, v in X_batch.items()}
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            probs = torch.sigmoid(logits)

            metric.update(probs, y_batch.int())

    return metric.compute()


def train(
    model,
    optimizer,
    loss_fn,
    train_metric,
    val_metric,
    train_loader,
    valid_loader,
    config,
    checkpoint_path="best_model.pth",
):

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        patience=config["sched_patience"],
        factor=config["sched_factor"]
    )

    history = {
        "train_losses": [],
        "train_metrics": [],
        "valid_metrics": [],
    }

    best_val_metric = 0.0

    start_epoch = config["start_epoch"]
    end_epoch = config["end_epoch"]

    for epoch in range(start_epoch, end_epoch):

        model.train()
        train_metric.reset()

        total_loss = 0.0

        for index, (X_batch, y_batch) in enumerate(train_loader):

            X_batch = {k: v.to(device) for k, v in X_batch.items()}
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = loss_fn(logits, y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])

            optimizer.step()

            total_loss += loss.item()

            probs = torch.sigmoid(logits)

            train_metric.update(
                probs.detach(),
                y_batch.int()
            )

            print(
                f"\rBatch {index+1}/{len(train_loader)} "
                f"loss={total_loss/(index+1):.4f}",
                end=""
            )

        # ----- Epoch metrics -----

        train_metric_value = train_metric.compute().item()
        val_metric_value = evaluate_tm(model, valid_loader, val_metric).item()

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric_value)
        history["valid_metrics"].append(val_metric_value)

        scheduler.step(val_metric_value)

        if val_metric_value > best_val_metric:

            best_val_metric = val_metric_value

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_metric": val_metric_value,
                    "config": config
                },
                checkpoint_path,
            )

        print(
            f"\rEpoch {epoch+1}/{end_epoch}, "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train metric: {train_metric_value:.2%}, "
            f"valid metric: {val_metric_value:.2%} "
            f"{'[BEST]' if val_metric_value == best_val_metric else ''}"
        )

        wandb.log({
            "train/loss": history['train_losses'][-1],
            "train/pr_auc": train_metric_value,
            "val/pr_auc": val_metric_value,
            "epoch": epoch+1
        }, step=epoch+1)

    return history

In [ ]:
from torchmetrics.classification import (
    MultilabelPrecisionRecallCurve,
    MultilabelF1Score,
    MultilabelAveragePrecision
)
import torch
import matplotlib.pyplot as plt
import numpy as np
from typing import Tuple, Dict


def plot_pr_auc_and_f1(model, valid_loader, device) -> Dict[str, float]:
    """
    Plot PR-AUC curves, F1 scores, and compute metrics for multi-label classification.

    Args:
        model: Trained PyTorch model
        valid_loader: Validation DataLoader
        device: Device to run inference on ('cuda' or 'cpu')

    Returns:
        Dict containing macro PR-AUC, best F1 thresholds (macro and per-label)
    """

    # =========================================================================
    # 1. COLLECT PREDICTIONS
    # =========================================================================

    model.eval()
    all_probs = []
    all_labels = []

    print("Collecting predictions from validation set...")

    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch = {k: v.to(device) for k, v in X_batch.items()}
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu())
            all_labels.append(y_batch.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    print(f"✓ Collected {all_probs.shape[0]} samples")

    # =========================================================================
    # 2. COMPUTE PR CURVES
    # =========================================================================

    pr_curve = MultilabelPrecisionRecallCurve(num_labels=6, thresholds=None)
    pr_curves = pr_curve(all_probs, all_labels.int())

    precision_per_label = pr_curves[0]  # List of 6 tensors
    recall_per_label = pr_curves[1]     # List of 6 tensors
    thresholds_per_label = pr_curves[2] # List of 6 tensors

    # =========================================================================
    # 3. PLOT PR CURVES PER LABEL
    # =========================================================================

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    label_names = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

    # Compute PR-AUC per label for display
    pr_auc_metric = MultilabelAveragePrecision(num_labels=6, average=None)
    pr_auc_per_label = pr_auc_metric(all_probs, all_labels.int())

    for i in range(6):
        recall = recall_per_label[i].numpy()
        precision = precision_per_label[i].numpy()

        sort_idx = np.argsort(recall)
        recall_sorted = recall[sort_idx]
        precision_sorted = precision[sort_idx]

        axes[i].plot(recall_sorted, precision_sorted, linewidth=2, color='blue')

        baseline = all_labels[:, i].float().mean().item()
        axes[i].axhline(y=baseline, color='red', linestyle='--',
                       label=f'Baseline={baseline:.3f}', alpha=0.7)

        axes[i].set_title(f'PR Curve: {label_names[i]}\nAUC={pr_auc_per_label[i]:.3f}',
                         fontsize=12, fontweight='bold')
        axes[i].set_xlabel('Recall', fontsize=10)
        axes[i].set_ylabel('Precision', fontsize=10)
        axes[i].grid(True, alpha=0.3)
        axes[i].legend(loc='best', fontsize=9)
        axes[i].set_xlim([0, 1])
        axes[i].set_ylim([0, 1.05])

    plt.suptitle('PR Curves per Label', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

    # =========================================================================
    # 4. PLOT MACRO-AVERAGED PR CURVE
    # =========================================================================

    recall_points = np.linspace(0, 1, 1000)
    interpolated_precisions = []

    for i in range(6):
        recall = recall_per_label[i].numpy()
        precision = precision_per_label[i].numpy()

        sort_idx = np.argsort(recall)
        recall_sorted = recall[sort_idx]
        precision_sorted = precision[sort_idx]

        interp_precision = np.interp(
            recall_points,
            recall_sorted,
            precision_sorted,
            left=precision_sorted[0],
            right=0
        )

        interpolated_precisions.append(interp_precision)

    macro_precision = np.mean(interpolated_precisions, axis=0)

    pr_auc_macro_metric = MultilabelAveragePrecision(num_labels=6, average='macro')
    macro_pr_auc = pr_auc_macro_metric(all_probs, all_labels.int())

    plt.figure(figsize=(10, 8))

    for i in range(6):
        recall = recall_per_label[i].numpy()
        precision = precision_per_label[i].numpy()
        sort_idx = np.argsort(recall)
        plt.plot(recall[sort_idx], precision[sort_idx],
                alpha=0.2, linewidth=1, color='gray')

    plt.plot(recall_points, macro_precision,
            linewidth=3, color='navy',
            label=f'Macro-Average (AUC={macro_pr_auc:.3f})')

    baseline_macro = all_labels.float().mean().item()
    plt.axhline(y=baseline_macro, color='red', linestyle='--',
               linewidth=2, label=f'Random Baseline={baseline_macro:.3f}')

    plt.title('Macro-Averaged PR Curve', fontsize=16, fontweight='bold')
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.legend(loc='best', fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.xlim([0, 1])
    plt.ylim([0, 1.05])
    plt.tight_layout()
    plt.show()

    # =========================================================================
    # 5. COMPUTE BEST F1 THRESHOLD AND SCORE PER LABEL (NEW!)
    # =========================================================================

    print("\nComputing best F1 threshold per label...")

    thresholds_f1 = torch.linspace(0, 1, 101)  # 101 points for finer granularity

    # Store results per label
    best_f1_per_label = {}
    best_threshold_per_label = {}
    f1_scores_per_label = {label: [] for label in label_names}

    # Compute F1 for each label separately
    for label_idx in range(6):
        label_name = label_names[label_idx]
        f1_scores = []

        for thresh in thresholds_f1:
            # Binarize predictions for this label
            preds_binary = (all_probs[:, label_idx] >= thresh.item()).int()
            labels_binary = all_labels[:, label_idx].int()

            # Calculate F1 for this label
            # F1 = 2 * (precision * recall) / (precision + recall)
            tp = ((preds_binary == 1) & (labels_binary == 1)).sum().float()
            fp = ((preds_binary == 1) & (labels_binary == 0)).sum().float()
            fn = ((preds_binary == 0) & (labels_binary == 1)).sum().float()

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
            f1_scores.append(f1.item() if torch.is_tensor(f1) else f1)

        # Find best threshold for this label
        best_idx = np.argmax(f1_scores)
        best_f1_per_label[label_name] = f1_scores[best_idx]
        best_threshold_per_label[label_name] = thresholds_f1[best_idx].item()
        f1_scores_per_label[label_name] = f1_scores

    # =========================================================================
    # 6. PLOT F1-SCORE VS THRESHOLD - MACRO (Existing)
    # =========================================================================

    print("\nComputing macro F1 scores across thresholds...")

    f1_scores_macro = []

    for thresh in thresholds_f1:
        f1_metric = MultilabelF1Score(
            num_labels=6,
            threshold=thresh.item(),
            average='macro'
        )
        f1 = f1_metric(all_probs, all_labels.int())
        f1_scores_macro.append(f1.item())

    best_idx_macro = np.argmax(f1_scores_macro)
    best_threshold_macro = thresholds_f1[best_idx_macro].item()
    best_f1_macro = f1_scores_macro[best_idx_macro]

    plt.figure(figsize=(10, 6))
    plt.plot(thresholds_f1.numpy(), f1_scores_macro, linewidth=2, color='green')

    plt.axvline(x=best_threshold_macro, color='red', linestyle='--',
               linewidth=2, label=f'Best Threshold={best_threshold_macro:.2f}')
    plt.scatter([best_threshold_macro], [best_f1_macro], color='red', s=100, zorder=5)

    plt.annotate(f'Max F1={best_f1_macro:.3f}',
                xy=(best_threshold_macro, best_f1_macro),
                xytext=(best_threshold_macro + 0.1, best_f1_macro - 0.05),
                fontsize=10,
                bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
                arrowprops=dict(arrowstyle='->', color='black'))

    plt.title('Macro F1-Score vs Classification Threshold',
             fontsize=14, fontweight='bold')
    plt.xlabel('Threshold', fontsize=12)
    plt.ylabel('Macro F1-Score', fontsize=12)
    plt.legend(loc='best', fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.xlim([0, 1])
    plt.tight_layout()
    plt.show()

    # =========================================================================
    # 7. PLOT F1-SCORE VS THRESHOLD - PER LABEL (NEW!)
    # =========================================================================

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for i, label_name in enumerate(label_names):
        # Plot F1 curve for this label
        axes[i].plot(thresholds_f1.numpy(), f1_scores_per_label[label_name],
                    linewidth=2, color='green')

        # Mark best threshold
        best_thresh = best_threshold_per_label[label_name]
        best_f1 = best_f1_per_label[label_name]

        axes[i].axvline(x=best_thresh, color='red', linestyle='--',
                       linewidth=2, alpha=0.7)
        axes[i].scatter([best_thresh], [best_f1], color='red', s=100, zorder=5)

        # Annotate
        axes[i].annotate(f'Max F1={best_f1:.3f}\n@{best_thresh:.2f}',
                        xy=(best_thresh, best_f1),
                        xytext=(best_thresh + 0.15, best_f1 - 0.1),
                        fontsize=9,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                        arrowprops=dict(arrowstyle='->', color='black', lw=1))

        axes[i].set_title(f'F1 vs Threshold: {label_name}',
                         fontsize=12, fontweight='bold')
        axes[i].set_xlabel('Threshold', fontsize=10)
        axes[i].set_ylabel('F1-Score', fontsize=10)
        axes[i].grid(True, alpha=0.3)
        axes[i].set_xlim([0, 1])
        axes[i].set_ylim([0, 1.05])

    plt.suptitle('F1-Score vs Threshold per Label', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

    # =========================================================================
    # 8. PLOT BEST F1 THRESHOLD COMPARISON (NEW!)
    # =========================================================================

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Best F1 scores
    labels_with_macro = label_names + ['Macro']
    f1_values = [best_f1_per_label[label] for label in label_names] + [best_f1_macro]

    colors_f1 = ['skyblue'] * 6 + ['navy']
    bars1 = ax1.bar(labels_with_macro, f1_values, color=colors_f1,
                    alpha=0.7, edgecolor='black')

    # Add value labels
    for bar, val in zip(bars1, f1_values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax1.set_title('Best F1-Score per Label', fontsize=14, fontweight='bold')
    ax1.set_ylabel('F1-Score', fontsize=12)
    ax1.set_xlabel('Label', fontsize=12)
    ax1.set_ylim([0, max(f1_values) * 1.15])
    ax1.grid(axis='y', alpha=0.3)
    ax1.tick_params(axis='x', rotation=45)

    # Plot 2: Best thresholds
    threshold_values = [best_threshold_per_label[label] for label in label_names] + [best_threshold_macro]

    bars2 = ax2.bar(labels_with_macro, threshold_values, color=colors_f1,
                    alpha=0.7, edgecolor='black')

    # Add value labels
    for bar, val in zip(bars2, threshold_values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Add reference line at 0.5
    ax2.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5,
                alpha=0.5, label='Default (0.5)')

    ax2.set_title('Best Classification Threshold per Label', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Threshold', fontsize=12)
    ax2.set_xlabel('Label', fontsize=12)
    ax2.set_ylim([0, 1.0])
    ax2.grid(axis='y', alpha=0.3)
    ax2.legend(loc='best')
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    # =========================================================================
    # 9. PLOT PR-AUC BAR CHART (Existing)
    # =========================================================================

    labels = label_names + ['Macro']
    values = list(pr_auc_per_label.numpy()) + [macro_pr_auc.item()]

    baselines = [all_labels[:, i].float().mean().item() for i in range(6)]
    baselines.append(baseline_macro)

    colors = ['green' if val > base else 'red'
             for val, base in zip(values, baselines)]
    colors[-1] = 'navy'

    plt.figure(figsize=(12, 7))
    bars = plt.bar(labels, values, color=colors, alpha=0.7, edgecolor='black')

    for i, (bar, val) in enumerate(zip(bars, values)):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    for i, baseline in enumerate(baselines[:-1]):
        plt.hlines(baseline, i-0.4, i+0.4, colors='red',
                  linestyles='--', linewidth=1.5, alpha=0.5)

    plt.title('PR-AUC per Label and Macro Average',
             fontsize=14, fontweight='bold')
    plt.ylabel('PR-AUC Score', fontsize=12)
    plt.xlabel('Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.ylim([0, 1.0])
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # =========================================================================
    # 10. PRINT SUMMARY STATISTICS (Enhanced)
    # =========================================================================

    print("\n" + "="*70)
    print("VALIDATION METRICS SUMMARY")
    print("="*70)

    print(f"\n{'MACRO METRICS':^70}")
    print("-"*70)
    print(f"Macro-Averaged PR-AUC:     {macro_pr_auc:.4f}")
    print(f"Best Macro F1 Threshold:   {best_threshold_macro:.3f}")
    print(f"Best Macro F1-Score:       {best_f1_macro:.4f}")

    print(f"\n{'PER-LABEL METRICS':^70}")
    print("-"*70)
    print(f"{'Label':<15} | {'PR-AUC':<8} | {'Best F1':<8} | {'Threshold':<10} | {'Baseline':<8} | {'Δ PR-AUC':<10}")
    print("-"*70)

    for i, label in enumerate(label_names):
        baseline = baselines[i]
        pr_auc_val = pr_auc_per_label[i].item()
        improvement = pr_auc_val - baseline
        best_f1 = best_f1_per_label[label]
        best_thresh = best_threshold_per_label[label]

        print(f"{label:<15} | {pr_auc_val:>6.4f}  | {best_f1:>6.4f}  | "
              f"{best_thresh:>8.3f}  | {baseline:>6.4f}  | {improvement:>+7.4f}")

    print("="*70 + "\n")

    # =========================================================================
    # 11. RETURN RESULTS (Enhanced)
    # =========================================================================

    results = {
        # Macro metrics
        'macro_pr_auc': macro_pr_auc.item(),
        'best_f1_threshold_macro': best_threshold_macro,
        'best_f1_score_macro': best_f1_macro,

        # Per-label PR-AUC
        'per_label_pr_auc': {
            label_names[i]: pr_auc_per_label[i].item()
            for i in range(6)
        },

        # Per-label best F1 (NEW!)
        'per_label_best_f1': best_f1_per_label,

        # Per-label best threshold (NEW!)
        'per_label_best_threshold': best_threshold_per_label,

        # Baselines
        'per_label_baseline': {
            label_names[i]: baselines[i]
            for i in range(6)
        }
    }

    return results

# Model Building, Training & Evaluation

## Stacked LSTM

In [ ]:
class StackedLSTMModel(nn.Module):
    def __init__(self, vocab_size=30522, embed_dim=256, n_layers=1,
                 hidden_dim=32, pad_id=0, dropout=0.0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 6)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, (h_n, c_n) = self.lstm(packed)
        return self.output(h_n[-1])

### Base Architecture & Hyperparameters

In [ ]:
base_config = {
    # Model architecture
    "model_type": "Stacked LSTM",
    "embed_dim": 256,
    "hidden_dim": 32,
    "num_layers": 1,
    "dropout": 0.0,

    # Training
    "learning_rate": 0.001,
    "batch_size": 256,
    "optimizer": "AdamW",
    "sched_patience" : 2,
    "sched_factor": 0.5,
    "max_grad_norm": 1.0,
    "start_epoch": 0,
    "end_epoch": 10
}

### Config_1 Training

In [ ]:
# Setup New Config
config = base_config

model = StackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-stacked-lstm-config-1",
                config=config,
                notes="base config"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_lstm_model.pth'
    )

### Config_2 Training

In [ ]:
# Setup New Config
config = base_config
config["hidden_dim"] = 16

model = StackedLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-stacked-lstm-config-2",
                config=config,
                notes="set hidden_dim=16"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_lstm_model.pth'
    )

In [ ]:
results = plot_pr_auc_and_f1(model, valid_loader, device)

    # Access macro metrics
    print(f"\nMacro PR-AUC: {results['macro_pr_auc']:.4f}")
    print(f"Macro Best F1: {results['best_f1_score_macro']:.4f} @ threshold {results['best_f1_threshold_macro']:.3f}")

    # Access per-label metrics
    print("\nPer-Label Best F1 Scores:")
    for label, f1 in results['per_label_best_f1'].items():
        threshold = results['per_label_best_threshold'][label]
        print(f"  {label}: F1={f1:.4f} @ threshold={threshold:.3f}")

    # Get optimal thresholds for deployment
    print("\nOptimal thresholds for classification:")
    for label, threshold in results['per_label_best_threshold'].items():
        print(f"  {label}: {threshold:.3f}")

## Stacked GRU

In [ ]:
class StackedGRUModel(nn.Module):
    def __init__(self, vocab_size=30522, embed_dim=256, n_layers=1,
                 hidden_dim=32, pad_id=0, dropout=0.0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 6)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, h_n  = self.gru(packed)
        return self.output(h_n[-1])

In [ ]:
base_config = {
    # Model architecture
    "model_type": "Stacked GRU",
    "embed_dim": 256,
    "hidden_dim": 32,
    "num_layers": 1,
    "dropout": 0.0,

    # Training
    "learning_rate": 0.001,
    "batch_size": 256,
    "optimizer": "AdamW",
    "sched_patience" : 2,
    "sched_factor": 0.5,
    "max_grad_norm": 1.0,
    "start_epoch": 0,
    "end_epoch": 10
}

### Config_1 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)

model = StackedGRUModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-stacked-gru-config-2",
                config=config,
                notes="set hidden_dim=16"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_gru_model.pth'
    )

## Stacked Bidirectional LSTM

In [ ]:
class StackedBiLSTMModel(nn.Module):
    def __init__(self, vocab_size=30522, embed_dim=256, n_layers=1,
                 hidden_dim=32, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.output = nn.Linear(2 * hidden_dim, 6)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, (h_n, c_n) = self.lstm(packed)

        n_dims = self.output.in_features # added
        top_states = h_n[-2:].permute(1, 0, 2).reshape(-1, n_dims) # added
        return self.output(top_states) # added

In [ ]:
base_config = {
    # Model architecture
    "model_type": "Stacked BiLSTM",
    "embed_dim": 256,
    "hidden_dim": 32,
    "num_layers": 1,
    "dropout": 0.0,

    # Training
    "learning_rate": 0.001,
    "batch_size": 256,
    "optimizer": "AdamW",
    "sched_patience" : 2,
    "sched_factor": 0.5,
    "max_grad_norm": 1.0,
    "start_epoch": 0,
    "end_epoch": 10
}

### Config_1 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)

model = StackedBiLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-stacked-bilstm-config-1",
                config=config,
                notes="base config"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_bilstm_model.pth'
    )

### Config_2 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)
config["num_layers"] = 2
config["dropout"] = 0.4

model = StackedBiLSTMModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-stacked-bilstm-config-2",
                config=config,
                notes="set num_layers=2, dropout=0.4"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_bilstm_model-2.pth'
    )

In [ ]:
results = plot_pr_auc_and_f1(model, valid_loader, device)

    # Access macro metrics
    print(f"\nMacro PR-AUC: {results['macro_pr_auc']:.4f}")
    print(f"Macro Best F1: {results['best_f1_score_macro']:.4f} @ threshold {results['best_f1_threshold_macro']:.3f}")

    # Access per-label metrics
    print("\nPer-Label Best F1 Scores:")
    for label, f1 in results['per_label_best_f1'].items():
        threshold = results['per_label_best_threshold'][label]
        print(f"  {label}: F1={f1:.4f} @ threshold={threshold:.3f}")

    # Get optimal thresholds for deployment
    print("\nOptimal thresholds for classification:")
    for label, threshold in results['per_label_best_threshold'].items():
        print(f"  {label}: {threshold:.3f}")

## Stacked Bidirectional GRU

In [ ]:
class StackedBiGRUModel(nn.Module):
    def __init__(self, vocab_size=30522, embed_dim=256, n_layers=1,
                 hidden_dim=32, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.output = nn.Linear(2 * hidden_dim, 6)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, h_n  = self.gru(packed)

        n_dims = self.output.in_features # added
        top_states = h_n[-2:].permute(1, 0, 2).reshape(-1, n_dims) # added
        return self.output(top_states) # added

In [ ]:
base_config = {
    # Model architecture
    "model_type": "Stacked BiGRU",
    "embed_dim": 256,
    "hidden_dim": 32,
    "num_layers": 1,
    "dropout": 0.0,

    # Training
    "learning_rate": 0.001,
    "batch_size": 256,
    "optimizer": "AdamW",
    "sched_patience" : 2,
    "sched_factor": 0.5,
    "max_grad_norm": 1.0,
    "start_epoch": 0,
    "end_epoch": 10
}

### Config_1 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)

model = StackedBiGRUModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-stacked-bigru-config-1",
                config=config,
                notes="base config"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_bigru_model.pth'
    )

### Config_2 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)
config["num_layers"] = 2
config["dropout"] = 0.4

model = StackedBiGRUModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-stacked-bigru-config-2",
                config=config,
                notes="set num_layers=2, dropout=0.4"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_bigru_model.pth'
    )

In [ ]:
results = plot_pr_auc_and_f1(model, valid_loader, device)

    # Access macro metrics
    print(f"\nMacro PR-AUC: {results['macro_pr_auc']:.4f}")
    print(f"Macro Best F1: {results['best_f1_score_macro']:.4f} @ threshold {results['best_f1_threshold_macro']:.3f}")

    # Access per-label metrics
    print("\nPer-Label Best F1 Scores:")
    for label, f1 in results['per_label_best_f1'].items():
        threshold = results['per_label_best_threshold'][label]
        print(f"  {label}: F1={f1:.4f} @ threshold={threshold:.3f}")

    # Get optimal thresholds for deployment
    print("\nOptimal thresholds for classification:")
    for label, threshold in results['per_label_best_threshold'].items():
        print(f"  {label}: {threshold:.3f}")

In [ ]:
# Train more 10 epochs:
config["start_epoch"] = 10
config["end_epoch"] = 20

with wandb.init(project="nontoxic-world",
                id="84bepott",
                config=config,
                resume="allow"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_bigru_model_2.pth'
    )

### Config_3 Training

In [ ]:
# Setup New Config
config = deepcopy(base_config)
config["num_layers"] = 2
config["dropout"] = 0.5
config["end_epoch"] = 5

model = StackedBiGRUModel(
    vocab_size=len(tokenizer),
    embed_dim=config["embed_dim"],
    n_layers=config["num_layers"],
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
).to(device)

# Training configuration
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])

# Metric (move to device!)
macro_pr_auc = MultilabelAveragePrecision(
    num_labels=6,
    average="macro"
).to(device)

# Train
with wandb.init(project="nontoxic-world",
                name="from-scratch-stacked-bigru-config-3",
                config=config,
                notes="set num_layers=2, dropout=0.5, 5 epochs without freezing embedding and 10 with freeze"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_bigru_model_3.pth'
    )

In [ ]:
for param in model.embed.parameters():
    param.requires_grad = False

# Train more 10 epochs:
config["start_epoch"] = 5
config["end_epoch"] = 10

with wandb.init(project="nontoxic-world",
                id="h8qg1wsu",
                config=config,
                resume="allow"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_bigru_model_4.pth'
    )

In [ ]:
results = plot_pr_auc_and_f1(model, valid_loader, device)

# Access macro metrics
print(f"\nMacro PR-AUC: {results['macro_pr_auc']:.4f}")
print(f"Macro Best F1: {results['best_f1_score_macro']:.4f} @ threshold {results['best_f1_threshold_macro']:.3f}")

# Access per-label metrics
print("\nPer-Label Best F1 Scores:")
for label, f1 in results['per_label_best_f1'].items():
    threshold = results['per_label_best_threshold'][label]
    print(f"  {label}: F1={f1:.4f} @ threshold={threshold:.3f}")

# Get optimal thresholds for deployment
print("\nOptimal thresholds for classification:")
for label, threshold in results['per_label_best_threshold'].items():
    print(f"  {label}: {threshold:.3f}")

In [ ]:

config["start_epoch"] = 10
config["end_epoch"] = 15

with wandb.init(project="nontoxic-world",
                id="h8qg1wsu",
                config=config,
                resume="allow"
               ):
    history = train(
        model,
        optimizer,
        xentropy,
        macro_pr_auc,
        macro_pr_auc,
        train_loader,
        valid_loader,
        config,
        checkpoint_path='stacked_bigru_model_5.pth'
    )

## Training Summery Graphs

![from-scratch-epochs.png](attachment:from-scratch-epochs.png)

![from-scratch-train-metric.png](attachment:from-scratch-train-metric.png)

![from-scratch-valid-metric.png](attachment:from-scratch-valid-metric.png)

# Testing

## Stacked GRU Model

In [ ]:
checkpoint_path = '../models/stacked_gru_model.pth'  # Your checkpoint file

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)

# Get model parameters
config = checkpoint.get('config', {})
vocab_size = 30_000
embed_dim = config.get('embed_dim', 256)
hidden_dim = config.get('hidden_dim', 128)
n_layers = config.get('num_layers', 2)
dropout = config.get('dropout', 0.2)

# Create model (use your model class from the notebook)
model = StackedGRUModel(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    n_layers=n_layers,
    hidden_dim=hidden_dim,
    dropout=dropout
).to(device)

# Load weights
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✓ Model loaded successfully!")

In [ ]:
results = plot_pr_auc_and_f1(model, test_loader, device)

# Access macro metrics
print(f"\nMacro PR-AUC: {results['macro_pr_auc']:.4f}")
print(f"Macro Best F1: {results['best_f1_score_macro']:.4f} @ threshold {results['best_f1_threshold_macro']:.3f}")

# Access per-label metrics
print("\nPer-Label Best F1 Scores:")
for label, f1 in results['per_label_best_f1'].items():
    threshold = results['per_label_best_threshold'][label]
    print(f"  {label}: F1={f1:.4f} @ threshold={threshold:.3f}")

# Get optimal thresholds for deployment
print("\nOptimal thresholds for classification:")
for label, threshold in results['per_label_best_threshold'].items():
    print(f"  {label}: {threshold:.3f}")

## Stacked Bidirectional GRU Model

In [ ]:
checkpoint_path = '../models/stacked_bigru_model.pth'  # Your checkpoint file

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)

# Get model parameters
config = checkpoint.get('config', {})
vocab_size = 30_000
embed_dim = config.get('embed_dim', 256)
hidden_dim = config.get('hidden_dim', 128)
n_layers = config.get('num_layers', 2)
dropout = config.get('dropout', 0.2)

# Create model (use your model class from the notebook)
model = StackedBiGRUModel(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    n_layers=n_layers,
    hidden_dim=hidden_dim,
    dropout=dropout
).to(device)

# Load weights
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✓ Model loaded successfully!")

In [ ]:
results = plot_pr_auc_and_f1(model, valid_loader, device)
    
# Access macro metrics
print(f"\nMacro PR-AUC: {results['macro_pr_auc']:.4f}")
print(f"Macro Best F1: {results['best_f1_score_macro']:.4f} @ threshold {results['best_f1_threshold_macro']:.3f}")

# Access per-label metrics
print("\nPer-Label Best F1 Scores:")
for label, f1 in results['per_label_best_f1'].items():
    threshold = results['per_label_best_threshold'][label]
    print(f"  {label}: F1={f1:.4f} @ threshold={threshold:.3f}")

# Get optimal thresholds for deployment
print("\nOptimal thresholds for classification:")
for label, threshold in results['per_label_best_threshold'].items():
    print(f"  {label}: {threshold:.3f}")